In [ ]:
"""
Copyright (c) 2021-2024 D-Robotics Corporation

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
"""

In [ ]:
!cat /proc/meminfo | grep Mem

In [ ]:
# 导入所需要的包import numpy as npimport cv2import timeimport osimport argparseimport hbm_runtimefrom typing import Dict, Optional# For displaying images in Jupyterimport matplotlib.pyplot as plt# ============== Utility Functions (extracted from utils) ==============def bgr_to_nv12_planes(image: np.ndarray) -> tuple:    """    Convert a BGR image to NV12 format (Y and UV planes).    """    height, width = image.shape[:2]    area = height * width    # Convert to planar YUV I420 format    yuv420p = cv2.cvtColor(image, cv2.COLOR_BGR2YUV_I420)    yuv420p = yuv420p.reshape((area * 3 // 2,))    # Extract Y, U, V planes    y = yuv420p[:area].reshape((height, width))    u = yuv420p[area:area + area // 4].reshape((height // 2, width // 2))    v = yuv420p[area + area // 4:].reshape((height // 2, width // 2))    # Interleave U and V to form UV plane    uv = np.stack((u, v), axis=-1)    # Add batch and channel dimensions    y = y[np.newaxis, :, :, np.newaxis]    uv = uv[np.newaxis, :, :, :]    return y, uvdef resized_image(img: np.ndarray, input_W: int, input_H: int,                  resize_type: int = 1,                  interpolation=cv2.INTER_NEAREST) -> np.ndarray:    """    Resize image with either direct resize or letterbox strategy.    """    img_h, img_w = img.shape[:2]    if resize_type == 0:  # Direct resize        resized = cv2.resize(img, (input_W, input_H), interpolation=interpolation)    elif resize_type == 1:  # Letterbox resize (preserve aspect ratio)        scale = min(input_H / img_h, input_W / img_w)        new_w, new_h = int(img_w * scale), int(img_h * scale)        resized = cv2.resize(img, (new_w, new_h))        pad_w = input_W - new_w        pad_h = input_H - new_h        left, right = pad_w // 2, pad_w - pad_w // 2        top, bottom = pad_h // 2, pad_h - pad_h // 2        # Pad image with gray (127,127,127)        resized = cv2.copyMakeBorder(resized, top, bottom, left, right,                                     borderType=cv2.BORDER_CONSTANT,                                     value=(127, 127, 127))    else:        raise ValueError(f"Invalid resize_type: {resize_type}, must be 0 or 1")    return resizeddef print_topk_predictions(output: np.ndarray,                           idx2label: dict = None,                           topk: int = 5) -> None:    """    Print top-k classification predictions.    """    # Softmax with stability adjustment    exp_logits = np.exp(output - np.max(output))    probabilities = exp_logits / np.sum(exp_logits)    # Top-k indices    topk_idx = np.argsort(probabilities)[-topk:][::-1]    topk_prob = probabilities[topk_idx]    print(f"Top-{topk} Predictions:")    for i in range(topk):        idx = topk_idx[i]        prob = topk_prob[i]        label = idx2label[idx] if idx2label and idx in idx2label else f"Class {idx}"        print(f"{label}: {prob:.4f}")def display_image(image_cv, title="Image", size=(5, 5)):    """Displays an OpenCV image (BGR) using Matplotlib (RGB)."""    # Convert BGR to RGB for Matplotlib    image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)    plt.figure(figsize=size)    plt.imshow(image_rgb)    plt.title(title)    plt.axis('off') # Hide axes    plt.show()# ============== End Utility Functions ==============

# EfficientNet-Lite2 模型推理演示

本notebook演示了如何使用HB_HBMRuntime API进行EfficientNet-Lite2模型的图像分类推理。

## 1. 模型和数据验证

首先检查模型文件和测试图像是否在预期位置可用。

In [ ]:
# --- Configuration ---NOTEBOOK_DIR = '.' MODEL_NAME = 'efficientnet_lite2_224x224_nv12.hbm'MODEL_PATH = os.path.join(NOTEBOOK_DIR, './model', MODEL_NAME)IMAGE_NAME = 'Scottish_deerhound.JPEG'  # or zebra_cls.jpg for ResNet/MobileNetIMAGE_PATH = os.path.join(NOTEBOOK_DIR, './data', IMAGE_NAME)# --- End Configuration ---print(f"Notebook directory: {os.path.abspath(NOTEBOOK_DIR)}")print(f"Model path: {os.path.abspath(MODEL_PATH)}")print(f"Image path: {os.path.abspath(IMAGE_PATH)}")# Check if files existif not os.path.exists(MODEL_PATH):    print(f"WARNING: Model file not found at {MODEL_PATH}")if not os.path.exists(IMAGE_PATH):    print(f"WARNING: Image file not found at {IMAGE_PATH}")    print("Using new HB_HBMRuntime API instead of legacy DNN API")

## 2. 模型类定义

定义EfficientNet-Lite2封装类，该类包含使用新的HB_HBMRuntime API进行模型加载、预处理、推理和后处理的所有步骤。

In [ ]:
# EfficientNet-Lite2 Classification Modelclass EfficientNetLite2:    """    @brief Wrapper class for running inference using an EfficientNet-Lite2 model through HB_HBMRuntime.    """    def __init__(self, model_path):        """        @brief Initialize the EfficientNet-Lite2 model with model path and extract I/O details.        """        # Load model runtime        self.model = hbm_runtime.HB_HBMRuntime(model_path)        # Retrieve model name and input/output names        self.model_name = self.model.model_names[0]        self.input_names = self.model.input_names[self.model_name]        self.output_names = self.model.output_names[self.model_name]        self.shapes = self.model.input_shapes[self.model_name]        # Extract input resolution (Height, Width)        self.input_H = self.shapes[self.input_names[0]][1]        self.input_W = self.shapes[self.input_names[0]][2]                print(f"Model loaded: {self.model_name}")        print(f"Input shape: {self.input_W} x {self.input_H}")        print(f"Input names: {self.input_names}")        print(f"Output names: {self.output_names}")    def pre_process(self, img: np.ndarray, resize_type: int = 1):        """        @brief Preprocess input image to match model input format.        """        # Resize and convert image to NV12 format        resize_img = resized_image(img, self.input_W, self.input_H, resize_type)        y, uv = bgr_to_nv12_planes(resize_img)        return {            self.model_name: {                self.input_names[0]: y,                self.input_names[1]: uv            }        }    def forward(self, input_tensor):        """        @brief Run forward inference using the preprocessed input tensor.        """        outputs = self.model.run(input_tensor)        return outputs[self.model_name]    def post_process(self, outputs, idx2label=None):        """        @brief Postprocess output and print top-K predicted labels.        """        # Display top-K predictions from output        print_topk_predictions(outputs[self.output_names[0]][0], idx2label)        return outputs[self.output_names[0]][0]print("EfficientNet-Lite2 model class defined successfully")

## 3. 模型加载和初始化

加载HBM模型文件并初始化EfficientNet-Lite2推理管道。此步骤将验证模型结构并提取输入/输出张量信息。

In [ ]:
print(f"Loading HB_HBMRuntime model from: {MODEL_PATH}")

try:
    # Initialize the EfficientNet-Lite2 model
    efficientnet = EfficientNetLite2(MODEL_PATH)
    print("Model initialization successful!")
    
    # Print detailed model information
    print(f"\n=== Model Details ===")
    print(f"Model count: {efficientnet.model.model_count}")
    print(f"Input tensor shapes: {efficientnet.model.input_shapes}")
    print(f"Output tensor shapes: {efficientnet.model.output_shapes}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the HB_HBMRuntime environment is available and the model path is correct.")
    efficientnet = None

# EfficientNet-Lite2 模型推理演示

本notebook演示了如何使用HB_HBMRuntime API进行EfficientNet-Lite2模型的图像分类推理。

## 1. 模型和数据验证

首先检查模型文件和测试图像是否在预期位置可用。

In [ ]:
print(f"Loading image from: {IMAGE_PATH}")
original_bgr_image = cv2.imread(IMAGE_PATH)

if original_bgr_image is None:
    print(f"Failed to load image at {IMAGE_PATH}. Please check the path.")
    preprocessed_input = None
else:
    print(f"Original image loaded. Shape: {original_bgr_image.shape}")
    display_image(original_bgr_image, title="Original Image")

    if efficientnet is not None:
        print(f"Preprocessing image for model input ({efficientnet.input_W}x{efficientnet.input_H})...")
        
        # Use the model's preprocessing method
        preprocessed_input = efficientnet.pre_process(original_bgr_image)
        
        # Display resized image for visual verification
        resized_img = resized_image(original_bgr_image, efficientnet.input_W, efficientnet.input_H)
        display_image(resized_img, title=f"Resized Image ({efficientnet.input_W}x{efficientnet.input_H})")
        
        print("Image preprocessing completed successfully!")
        print(f"Preprocessed input structure: {list(preprocessed_input.keys())}")
    else:
        print("Skipping preprocessing - model not loaded")
        preprocessed_input = None

## 5. 模型推理

使用预处理后的输入通过EfficientNet-Lite2模型进行前向推理。此步骤测量推理时间并验证输出张量形状。

In [ ]:
if efficientnet is not None and preprocessed_input is not None:
    print("Running model inference using HB_HBMRuntime...")
    start_time_inference = time.time()
    try:
        # Run inference using the model's forward method
        outputs = efficientnet.forward(preprocessed_input)
        inference_time = time.time() - start_time_inference
        print(f"Inference completed in {inference_time:.4f} seconds.")
        print(f"Output shapes: {[(name, arr.shape) for name, arr in outputs.items()]}")
        
        # Store inference results for post-processing
        inference_successful = True
    except Exception as e:
        print(f"Error during inference: {e}")
        outputs = None
        inference_successful = False
else:
    print("Skipping inference - model or preprocessed input not available.")
    outputs = None
    inference_successful = False

## 6. 后处理和结果

处理模型输出以生成分类预测：
- 应用softmax将logits转换为概率
- 提取置信度最高的Top-K预测结果
- 准备结果以进行可视化

In [ ]:
if inference_successful and outputs is not None:
    print("\nStarting post-processing using Python implementation...")
    t0 = time.time()
    
    # Use the model's built-in post-processing
    try:
        classification_output = efficientnet.post_process(outputs)
        t1 = time.time()
        print(f"Post-processing completed in {(t1 - t0):.4f} seconds")
        
        # Get top prediction for image annotation
        exp_logits = np.exp(classification_output - np.max(classification_output))
        probabilities = exp_logits / np.sum(exp_logits)
        top_idx = np.argmax(probabilities)
        top_prob = probabilities[top_idx]
        
        print(f"\nTop prediction: Class {top_idx} with confidence {top_prob:.4f}")
        
    except Exception as e:
        print(f"Error during post-processing: {e}")
        classification_output = None
        
else:
    print("Skipping post-processing - inference was not successful.")
    classification_output = None

## 7. 结果可视化

显示最终分类结果：
- 按置信度排序的Top-5预测结果
- 标注了最高预测结果的原始图像
- 完整的推理管道摘要

In [ ]:
if classification_output is not None and original_bgr_image is not None:
    print("\n" + "=" * 10, "Final Results Display", "=" * 10)
    
    # Get probabilities for visualization
    exp_logits = np.exp(classification_output - np.max(classification_output))
    probabilities = exp_logits / np.sum(exp_logits)
    
    # Get top 5 predictions for detailed display
    top5_idx = np.argsort(probabilities)[-5:][::-1]
    
    print("Top 5 Classification Results:")
    for i, idx in enumerate(top5_idx):
        prob = probabilities[idx]
        print(f"  {i+1}. Class {idx}: {prob:.4f} confidence")
    
    # Annotate original image with top prediction
    top_idx = top5_idx[0]
    top_prob = probabilities[top_idx]
    display_text = f"Class {top_idx} ({top_prob:.3f})"
    
    annotated_image = original_bgr_image.copy()
    cv2.putText(annotated_image, display_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(annotated_image, "EfficientNet-Lite2 Prediction", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
    
    display_image(annotated_image, title="Classification Result")
    
else:
    print("No results to display - classification was not successful.")

print("\n--- EfficientNet-Lite2 Inference Complete ---")

In [ ]:
!hrt_model_exec perf --model_file ./model/efficientnet_lite2_260x260_nv12.hbm \
                    --core_id=0 \
                    --frame_count=200 \
                    --perf_time=0 \
                    --thread_num=3